In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import torch
from tqdm import tqdm

In [ ]:
!pip -q install facenet-pytorch

In [ ]:
from facenet_pytorch import MTCNN
print("MTCNN imported successfully")

In [ ]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

In [ ]:
!pip uninstall -y facenet-pytorch
!pip install -q facenet-pytorch==2.6.0

In [ ]:
from facenet_pytorch import MTCNN

print("MTCNN imported successfully")

In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/Deepfake_Project"

INPUT_PATH = os.path.join(PROJECT_PATH, "informative_frames")
OUTPUT_PATH = os.path.join(PROJECT_PATH, "aligned_faces")

os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
os.makedirs(os.path.join(OUTPUT_PATH, "original"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_PATH, "Deepfakes"), exist_ok=True)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

mtcnn = MTCNN(
    image_size=224,
    margin=20,
    keep_all=True,
    post_process=False,
    device=device
)

print(device)

In [ ]:
import os
import cv2

sample_video = os.listdir(os.path.join(INPUT_PATH, "original"))[0]
sample_frame = os.listdir(os.path.join(INPUT_PATH, "original", sample_video))[0]

image_path = os.path.join(
    INPUT_PATH,
    "original",
    sample_video,
    sample_frame
)

print(image_path)

image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

In [ ]:
boxes, probs = mtcnn.detect(image)

print("Boxes:", boxes)
print("Confidence:", probs)

In [ ]:
if boxes is None:
    print("No face detected")
else:
    print(f"{len(boxes)} face(s) detected")

In [ ]:
if boxes is not None:

    areas = []

    for box in boxes:
        x1, y1, x2, y2 = box
        areas.append((x2-x1)*(y2-y1))

    idx = areas.index(max(areas))
    best_box = boxes[idx]

    print(best_box)

In [ ]:
x1, y1, x2, y2 = best_box

x1 = max(0, int(x1))
y1 = max(0, int(y1))
x2 = min(image.shape[1], int(x2))
y2 = min(image.shape[0], int(y2))

face = image[y1:y2, x1:x2]

print(face.shape)

In [ ]:
aligned_face = cv2.resize(face, (224,224))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5,5))
plt.imshow(aligned_face)
plt.axis("off")
plt.title("Aligned Face")
plt.show()

In [ ]:
import os
import cv2

save_folder = os.path.join(OUTPUT_PATH, "original", sample_video)
os.makedirs(save_folder, exist_ok=True)

save_path = os.path.join(save_folder, sample_frame)

cv2.imwrite(save_path, cv2.cvtColor(aligned_face, cv2.COLOR_RGB2BGR))

print("Saved:", save_path)

In [ ]:
def process_dataset(input_root, output_root):

    processed = 0
    skipped = 0

    classes = ["original", "Deepfakes"]

    for cls in classes:

        input_class = os.path.join(input_root, cls)
        output_class = os.path.join(output_root, cls)

        videos = sorted(os.listdir(input_class))

        for video in tqdm(videos, desc=f"Processing {cls}"):

            input_video = os.path.join(input_class, video)
            output_video = os.path.join(output_class, video)

            os.makedirs(output_video, exist_ok=True)

            frames = sorted(os.listdir(input_video))

            for frame in frames:

                image_path = os.path.join(input_video, frame)

                image = cv2.imread(image_path)

                if image is None:
                    skipped += 1
                    continue

                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

                boxes, probs = mtcnn.detect(image)

                if boxes is None:
                    skipped += 1
                    continue

                areas = []

                for box in boxes:

                    x1, y1, x2, y2 = box

                    areas.append((x2-x1)*(y2-y1))

                idx = areas.index(max(areas))

                x1, y1, x2, y2 = boxes[idx]

                x1 = max(0, int(x1))
                y1 = max(0, int(y1))
                x2 = min(image.shape[1], int(x2))
                y2 = min(image.shape[0], int(y2))

                face = image[y1:y2, x1:x2]

                if face.size == 0:
                    skipped += 1
                    continue

                aligned = cv2.resize(face, (224,224))

                save_path = os.path.join(output_video, frame)

                cv2.imwrite(
                    save_path,
                    cv2.cvtColor(aligned, cv2.COLOR_RGB2BGR)
                )

                processed += 1

    print("="*50)
    print("Processing Completed")
    print("="*50)
    print("Processed :", processed)
    print("Skipped   :", skipped)

In [ ]:
process_dataset(
    INPUT_PATH,
    OUTPUT_PATH
)